In [ ]:
# Google Colab Notebook: Data Collection for Masked IP Detection
# Run this in Google Colab

"""
MASKED IP DETECTION - DATA COLLECTION
======================================
This notebook collects IP data from various public sources

Steps:
1. Install required packages
2. Mount Google Drive
3. Collect data from multiple sources
4. Save to Drive
"""

# ============================================================================
# CELL 1: Setup and Installation
# ============================================================================

!pip install -q geoip2 maxminddb python-whois ipwhois requests pandas tqdm

# ============================================================================
# CELL 2: Mount Google Drive
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

# Create project directory
import os
project_dir = '/content/drive/MyDrive/masked_ip_detection'
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f'{project_dir}/data/raw', exist_ok=True)

print(f"Project directory created: {project_dir}")

# ============================================================================
# CELL 3: Data Collector Class
# ============================================================================

import requests
import pandas as pd
from datetime import datetime
import time
from tqdm.notebook import tqdm
import ipaddress

class IPDataCollector:
    """Collect IP data from public sources"""
    
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def collect_tor_nodes(self):
        """Collect Tor exit nodes"""
        print("Collecting Tor exit nodes...")
        
        sources = [
            "https://check.torproject.org/exit-addresses",
            "https://www.dan.me.uk/torlist/",
        ]
        
        tor_ips = set()
        
        for url in tqdm(sources, desc="Tor sources"):
            try:
                response = requests.get(url, timeout=15)
                if response.status_code == 200:
                    lines = response.text.split('\n')
                    for line in lines:
                        if 'ExitAddress' in line:
                            ip = line.split()[1]
                            tor_ips.add(ip)
                        elif line.strip() and not line.startswith('#'):
                            parts = line.strip().split()
                            if parts and self._is_valid_ip(parts[0]):
                                tor_ips.add(parts[0])
                time.sleep(1)
            except Exception as e:
                print(f"Error: {e}")
        
        df = pd.DataFrame({
            'ip': list(tor_ips),
            'label': 1,  # Masked
            'type': 'tor',
            'collected_at': datetime.now()
        })
        
        return df
    
    def collect_proxy_lists(self):
        """Collect public proxy IPs"""
        print("Collecting proxy servers...")
        
        sources = [
            "https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/http.txt",
            "https://raw.githubusercontent.com/clarketm/proxy-list/master/proxy-list-raw.txt",
            "https://raw.githubusercontent.com/ShiftyTR/Proxy-List/master/proxy.txt",
            "https://raw.githubusercontent.com/monosans/proxy-list/main/proxies/http.txt",
        ]
        
        proxy_ips = set()
        
        for url in tqdm(sources, desc="Proxy sources"):
            try:
                response = requests.get(url, timeout=15)
                if response.status_code == 200:
                    lines = response.text.split('\n')
                    for line in lines:
                        if ':' in line:
                            ip = line.split(':')[0].strip()
                            if self._is_valid_ip(ip):
                                proxy_ips.add(ip)
                time.sleep(1)
            except Exception as e:
                print(f"Error: {e}")
        
        df = pd.DataFrame({
            'ip': list(proxy_ips),
            'label': 1,
            'type': 'proxy',
            'collected_at': datetime.now()
        })
        
        return df
    
    def collect_vpn_data(self):
        """Collect VPN provider data"""
        print("Collecting VPN provider info...")
        
        vpn_data = [
            {'asn': 'AS202795', 'provider': 'NordVPN', 'label': 1, 'type': 'vpn'},
            {'asn': 'AS43350', 'provider': 'NordVPN', 'label': 1, 'type': 'vpn'},
            {'asn': 'AS396356', 'provider': 'ExpressVPN', 'label': 1, 'type': 'vpn'},
            {'asn': 'AS328543', 'provider': 'Surfshark', 'label': 1, 'type': 'vpn'},
            {'asn': 'AS62371', 'provider': 'ProtonVPN', 'label': 1, 'type': 'vpn'},
            {'asn': 'AS32780', 'provider': 'IPVanish', 'label': 1, 'type': 'vpn'},
            {'asn': 'AS396982', 'provider': 'CyberGhost', 'label': 1, 'type': 'vpn'},
        ]
        
        df = pd.DataFrame(vpn_data)
        df['collected_at'] = datetime.now()
        
        return df
    
    def collect_datacenter_ips(self):
        """Collect datacenter ASN info"""
        print("Collecting datacenter info...")
        
        dc_data = [
            {'asn': 'AS16509', 'provider': 'AWS', 'label': 1, 'type': 'datacenter'},
            {'asn': 'AS14618', 'provider': 'AWS', 'label': 1, 'type': 'datacenter'},
            {'asn': 'AS15169', 'provider': 'Google Cloud', 'label': 1, 'type': 'datacenter'},
            {'asn': 'AS8075', 'provider': 'Azure', 'label': 1, 'type': 'datacenter'},
            {'asn': 'AS14061', 'provider': 'DigitalOcean', 'label': 1, 'type': 'datacenter'},
            {'asn': 'AS20473', 'provider': 'Vultr', 'label': 1, 'type': 'datacenter'},
            {'asn': 'AS16276', 'provider': 'OVH', 'label': 1, 'type': 'datacenter'},
        ]
        
        df = pd.DataFrame(dc_data)
        df['collected_at'] = datetime.now()
        
        return df
    
    def generate_legitimate_samples(self, n=5000):
        """Generate legitimate IP samples"""
        print(f"Generating {n} legitimate IP samples...")
        
        # Common ISP ASNs (residential)
        legitimate_asns = [
            'AS7922',  # Comcast
            'AS20115', # Charter
            'AS7018',  # AT&T
            'AS701',   # Verizon
            'AS3356',  # Level3
        ]
        
        data = []
        for asn in legitimate_asns:
            for i in range(n // len(legitimate_asns)):
                data.append({
                    'asn': asn,
                    'label': 0,  # Legitimate
                    'type': 'legitimate',
                    'collected_at': datetime.now()
                })
        
        return pd.DataFrame(data)
    
    @staticmethod
    def _is_valid_ip(ip):
        try:
            ipaddress.ip_address(ip)
            return True
        except:
            return False

# ============================================================================
# CELL 4: Collect All Data
# ============================================================================

collector = IPDataCollector(f'{project_dir}/data/raw')

# Collect from all sources
datasets = {}

datasets['tor'] = collector.collect_tor_nodes()
datasets['proxy'] = collector.collect_proxy_lists()
datasets['vpn'] = collector.collect_vpn_data()
datasets['datacenter'] = collector.collect_datacenter_ips()
datasets['legitimate'] = collector.generate_legitimate_samples(5000)

# ============================================================================
# CELL 5: Save and Summary
# ============================================================================

# Save each dataset
for name, df in datasets.items():
    filepath = f'{project_dir}/data/raw/{name}_data.csv'
    df.to_csv(filepath, index=False)
    print(f"Saved {name}: {len(df)} records to {filepath}")

# Create combined dataset
all_data = pd.concat([df for df in datasets.values()], ignore_index=True)
all_data.to_csv(f'{project_dir}/data/raw/combined_raw_data.csv', index=False)

print("\n" + "="*60)
print("DATA COLLECTION SUMMARY")
print("="*60)
print(f"Total records collected: {len(all_data)}")
print(f"\nBreakdown by type:")
print(all_data['type'].value_counts())
print(f"\nLabels distribution:")
print(f"Masked IPs (1): {(all_data['label'] == 1).sum()}")
print(f"Legitimate IPs (0): {(all_data['label'] == 0).sum()}")
print("="*60)

# ============================================================================
# CELL 6: Visualize Data Distribution
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Type distribution
all_data['type'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Distribution by IP Type')
axes[0].set_xlabel('Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Label distribution
all_data['label'].value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('Distribution by Label')
axes[1].set_xlabel('Label (0=Legitimate, 1=Masked)')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Legitimate', 'Masked'], rotation=0)

plt.tight_layout()
plt.savefig(f'{project_dir}/data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Data collection complete!")
print(f"✓ Files saved to: {project_dir}/data/raw/")